In [1]:
import torch
from fontTools.varLib import drop_implied_oncurve_points
from torch import nn
from d2l import torch as d2l

In [10]:
def dropout_layer(X,dropout):
    assert 0 <= dropout <= 1
    if dropout == 0:
        return X
    if dropout == 1:
        return torch.zeros_like(X)
    mask = (torch.rand(X.shape) > dropout).float()
    return X * mask / (1 - dropout)

In [11]:
X = torch.arange(16,dtype=torch.float32).reshape((2,8))
dropout_layer(X,0)

tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11., 12., 13., 14., 15.]])

In [12]:
dropout_layer(X,1)

tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [13]:
dropout_layer(X,0.5)

tensor([[ 0.,  0.,  0.,  6.,  8.,  0.,  0., 14.],
        [16.,  0., 20.,  0.,  0.,  0., 28.,  0.]])

In [14]:
dropout1, dropout2 = 0.2, 0.5
class Net(nn.Module):
    def __init__(self,num_inputs,num_outputs,num_hidden1,num_hidden2,is_train=True):
        super(Net, self).__init__()
        self.num_inputs = num_inputs
        self.training = is_train
        self.lin1 = nn.Linear(num_inputs,num_hidden1)
        self.lin2 = nn.Linear(num_hidden1,num_hidden2)
        self.lin3 = nn.Linear(num_hidden2,num_outputs)
        self.relu = nn.ReLU()

    def forward(self,X):
        H1 = self.relu(self.lin1(X.reshape((-1,self.num_inputs))))
        if self.training:
            H1 =  dropout_layer(H1,dropout1)
        H2 = self.relu(self.lin2(H1))
        if self.training:
            H2 = dropout_layer(H2,dropout2)
        out = self.lin3(H2)
        return out

In [16]:
num_inputs=784
num_outputs=10
num_hidden1=256
num_hidden2=256

In [18]:
net = Net(num_inputs,num_outputs,num_hidden1,num_hidden2)

In [19]:
num_epochs, lr, batch_size = 10, 0.5, 256
loss = nn.CrossEntropyLoss()
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)
trainer = torch.optim.SGD(net.parameters(), lr=lr)
d2l.train_ch3(net,train_iter,test_iter,loss,num_epochs,trainer)

In [22]:
"""
api实现暂退法
"""
net = nn.Sequential(nn.Flatten(),
                    nn.Linear(784,256),
                    nn.ReLU(),
                    nn.Dropout(dropout1),
                    nn.Linear(256,256),
                    nn.ReLU(),
                    nn.Dropout(dropout2),
                    nn.Linear(256,10))
def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight,std=0.01)  # 标准差是0.01

net.apply(init_weights)  # 同时展示层级

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=256, bias=True)
  (2): ReLU()
  (3): Dropout(p=0.2, inplace=False)
  (4): Linear(in_features=256, out_features=256, bias=True)
  (5): ReLU()
  (6): Dropout(p=0.5, inplace=False)
  (7): Linear(in_features=256, out_features=10, bias=True)
)

In [24]:
trainer = torch.optim.SGD(net.parameters(), lr=lr)
d2l.train_ch3(net,train_iter,test_iter,loss,num_epochs,trainer)